# 15 — ChemBERTa-2 on Tox21: Results & Token Importance

ChemBERTa-2 (Ahmad et al., 2022) is a RoBERTa-based transformer pre-trained on  
77M PubChem SMILES using a multi-task regression (MTR) objective.  
Fine-tuned here on the 12-task Tox21 benchmark.

**Sections**
1. Setup
2. Load model + evaluate on test set
3. 4-model comparison table (ECFP+XGB, SMILESGNN, AttentiveFP, ChemBERTa-2)
4. Token importance visualisation (gradient-based)
5. Best / worst task analysis
6. Summary

## 1. Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

from src.chemberta_model import create_chemberta_model
from src.datasets import load_dataset, get_task_config
from src.graph_train import MaskedMultiTaskLoss

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DIR = project_root / "models" / "tox21_chemberta_model"
CKPT_NAME = "DeepChem/ChemBERTa-77M-MTR"

print(f"Device    : {DEVICE}")
print(f"Model dir : {MODEL_DIR}")

## 2. Load Model & Test Data

In [ ]:
task_config = get_task_config("tox21")

# Rebuild architecture and load weights
model = create_chemberta_model(
    pretrained_model = CKPT_NAME,
    num_tasks        = 12,
    dropout          = 0.1,
).to(DEVICE)
model.load_state_dict(
    torch.load(MODEL_DIR / "best_model.pt", map_location=DEVICE, weights_only=True)
)
model.eval()
print("Model loaded.")

# Tokenizer (saved during training)
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR / "tokenizer"))
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

In [ ]:
# Load test split
_, _, test_df = load_dataset(
    "tox21", cache_dir=str(project_root / "data"),
    split_type="scaffold", seed=42,
)
print(f"Test compounds: {len(test_df)}")

class Tox21SMILESDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            smiles_list, padding="max_length", truncation=True,
            max_length=max_length, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

test_labels = test_df[task_config.task_names].values.astype(np.float32)
test_ds     = Tox21SMILESDataset(test_df["smiles"].tolist(), test_labels, tokenizer)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate(model, loader, device, task_config):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["input_ids"].to(device),
                           batch["attention_mask"].to(device))
            all_logits.append(logits.cpu())
            all_labels.append(batch["labels"])
    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs  = 1 / (1 + np.exp(-all_logits))

    per_task_auc, per_task_pr = {}, {}
    for t, name in enumerate(task_config.task_names):
        valid = ~np.isnan(all_labels[:, t])
        if valid.sum() < 2 or len(np.unique(all_labels[valid, t])) < 2:
            continue
        per_task_auc[name] = roc_auc_score(all_labels[valid, t], all_probs[valid, t])
        per_task_pr[name]  = average_precision_score(all_labels[valid, t], all_probs[valid, t])

    return {
        "mean_auc_roc":     np.mean(list(per_task_auc.values())),
        "mean_pr_auc":      np.mean(list(per_task_pr.values())),
        "per_task_auc_roc": per_task_auc,
        "per_task_pr_auc":  per_task_pr,
        "num_valid_tasks":  len(per_task_auc),
    }

test_metrics = evaluate(model, test_loader, DEVICE, task_config)
print(f"Test Mean AUC-ROC : {test_metrics['mean_auc_roc']:.4f}")
print(f"Test Mean PR-AUC  : {test_metrics['mean_pr_auc']:.4f}")
print(f"Valid tasks       : {test_metrics['num_valid_tasks']}/12")

## 3. Four-Model Comparison Table

In [ ]:
ecfp_auc = {
    "NR-AR": 0.7166, "NR-AR-LBD": 0.7954, "NR-AhR": 0.8177,
    "NR-Aromatase": 0.7390, "NR-ER": 0.7096, "NR-ER-LBD": 0.6963,
    "NR-PPAR-gamma": 0.6448, "SR-ARE": 0.7220, "SR-ATAD5": 0.6779,
    "SR-HSE": 0.6551, "SR-MMP": 0.7617, "SR-p53": 0.6862,
}
smilesgnn_auc = {
    "NR-AR": 0.7130, "NR-AR-LBD": 0.8301, "NR-AhR": 0.7819,
    "NR-Aromatase": 0.7025, "NR-ER": 0.6870, "NR-ER-LBD": 0.7081,
    "NR-PPAR-gamma": 0.6447, "SR-ARE": 0.7357, "SR-ATAD5": 0.6957,
    "SR-HSE": 0.6768, "SR-MMP": 0.7773, "SR-p53": 0.6933,
}
afp_auc = {
    "NR-AR": 0.7148, "NR-AR-LBD": 0.8325, "NR-AhR": 0.7970,
    "NR-Aromatase": 0.7149, "NR-ER": 0.7246, "NR-ER-LBD": 0.7000,
    "NR-PPAR-gamma": 0.6903, "SR-ARE": 0.6693, "SR-ATAD5": 0.7080,
    "SR-HSE": 0.7205, "SR-MMP": 0.8067, "SR-p53": 0.6941,
}
chemberta_auc = test_metrics["per_task_auc_roc"]

tasks = task_config.task_names
df_cmp = pd.DataFrame({
    "Task":        tasks,
    "ECFP4+XGB":   [ecfp_auc.get(t, np.nan) for t in tasks],
    "SMILESGNN":   [smilesgnn_auc.get(t, np.nan) for t in tasks],
    "AttentiveFP": [afp_auc.get(t, np.nan) for t in tasks],
    "ChemBERTa-2": [chemberta_auc.get(t, np.nan) for t in tasks],
}).set_index("Task")

summary = pd.DataFrame([{
    "ECFP4+XGB":   0.7052,
    "SMILESGNN":   0.7284,
    "AttentiveFP": 0.7311,
    "ChemBERTa-2": test_metrics["mean_auc_roc"],
}], index=["MEAN"])
df_cmp = pd.concat([df_cmp, summary])

print("4-Model AUC-ROC Comparison (Tox21 scaffold test set)")
print("=" * 65)
display(df_cmp.style
    .format("{:.4f}", na_rep="—")
    .highlight_max(axis=1, color="#d4edda")
)

In [ ]:
# Bar chart: Mean AUC-ROC across models
models = ["ECFP4+XGB", "SMILESGNN", "AttentiveFP", "ChemBERTa-2"]
aucs   = [0.7052, 0.7284, 0.7311, test_metrics["mean_auc_roc"]]
colors = ["#6c757d", "#007bff", "#28a745", "#dc3545"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(models, aucs, color=colors, width=0.5, edgecolor="white")
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=11)
ax.set_ylim(0.65, max(aucs) + 0.05)
ax.set_ylabel("Mean AUC-ROC (Tox21 test set)")
ax.set_title("Tox21 Model Comparison — Mean AUC-ROC", fontsize=13)
ax.axhline(0.7311, color="#28a745", linestyle="--", linewidth=0.8, label="AttentiveFP")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(project_root / "models" / "tox21_chemberta_model" / "model_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 4. Token Importance Visualisation

In [ ]:
def plot_token_importance(smiles, tokens, importance, title="", ax=None):
    """Horizontal bar chart coloured by gradient-based token importance."""
    imp = importance.copy()
    if imp.max() > imp.min():
        imp = (imp - imp.min()) / (imp.max() - imp.min())
    cmap   = cm.get_cmap("RdYlGn_r")
    colors = [cmap(float(v)) for v in imp]

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, max(3, len(tokens) * 0.35)))
    ax.barh(range(len(tokens)), imp, color=colors)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Normalised importance")
    ax.set_title(title, fontsize=9)
    return ax

print("Token importance helper defined.")

In [ ]:
# Visualise token importance for toxic compounds across best task
best_task = max(test_metrics["per_task_auc_roc"], key=test_metrics["per_task_auc_roc"].get)
task_idx  = task_config.task_names.index(best_task)
auc_val   = test_metrics["per_task_auc_roc"][best_task]

toxic_smiles = test_df[test_df[best_task] == 1]["smiles"].tolist()[:4]
print(f"Best task : {best_task}  (AUC={auc_val:.4f})")
print(f"Toxic compounds to explain: {len(toxic_smiles)}")

fig, axes = plt.subplots(1, len(toxic_smiles),
                          figsize=(5 * len(toxic_smiles), 6))
if len(toxic_smiles) == 1:
    axes = [axes]

for ax, smi in zip(axes, toxic_smiles):
    tokens, imp = model.get_token_importance(
        smi, tokenizer, task_idx=task_idx, device=DEVICE
    )
    plot_token_importance(
        smi, tokens, imp,
        title=f"{smi[:25]}{'...' if len(smi)>25 else ''}",
        ax=ax,
    )

fig.suptitle(f"Token Importance — {best_task} (AUC={auc_val:.4f})",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(project_root / "models" / "tox21_chemberta_model" / "token_importance.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 5. Best / Worst Task Analysis

In [ ]:
sorted_tasks = sorted(test_metrics["per_task_auc_roc"].items(),
                      key=lambda x: x[1], reverse=True)
print("Per-task AUC-ROC — ChemBERTa-2")
print("=" * 42)
for task, auc in sorted_tasks:
    bar = "█" * int(auc * 30)
    print(f"  {task:<20} {auc:.4f}  {bar}")

best_t  = sorted_tasks[0]
worst_t = sorted_tasks[-1]
print(f"\nBest  : {best_t[0]}  ({best_t[1]:.4f})")
print(f"Worst : {worst_t[0]}  ({worst_t[1]:.4f})")

## 6. Summary

In [ ]:
summary_data = {
    "Model":            ["ECFP4+XGBoost", "SMILESGNN", "AttentiveFP", "ChemBERTa-2"],
    "Mean AUC-ROC":     [0.7052, 0.7284, 0.7311, test_metrics["mean_auc_roc"]],
    "Mean PR-AUC":      [0.2962, 0.2685, 0.3164, test_metrics["mean_pr_auc"]],
    "Pre-training":     ["None", "None", "None", "77M PubChem SMILES (MTR)"],
    "Input type":       ["ECFP4 bits", "Graph + SMILES", "Graph", "SMILES tokens"],
    "Interpretability": ["SHAP (faithful)", "Post-hoc (compensation)",
                         "GradCAM (intrinsic)", "Gradient×token (intrinsic)"],
}
df_sum = pd.DataFrame(summary_data).set_index("Model")
display(df_sum.style.format({"Mean AUC-ROC": "{:.4f}", "Mean PR-AUC": "{:.4f}"}))

## Takeaways

| Finding | Detail |
|---|---|
| Pre-training impact | ChemBERTa-2 leverages 77M molecules — largest training set of all models |
| Interpretability | Gradient×token gives SMILES-level attribution without post-hoc approximation |
| Next steps | Ensemble (all 4 models) · UniMol (3D) · task-specific fine-tuning |
